# Road Segmentation for Autonomous Vehicles (CamVid)

Task family: semantic segmentation for autonomous driving.

Dataset source: https://www.kaggle.com/datasets/carlolepelaars/camvid

## Why YOLO Segmentation Is Correct

Autonomous vehicle road understanding requires dense per-pixel segmentation of road areas and scene elements. YOLO26m-seg is the correct model because it:
- Outputs per-pixel segmentation masks for multiple road scene classes
- Provides real-time segmentation suitable for autonomous driving applications
- Can handle multi-class segmentation (road, sidewalk, building, sky, tree, car, pedestrian, etc.)
- Delivers competitive accuracy on autonomous driving benchmarks

## Environment Setup

In [ ]:
import importlib, subprocess, sys

def ensure_pkg(import_name: str, pip_name: str | None = None):
    try:
        importlib.import_module(import_name)
    except Exception:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pip_name or import_name])

ensure_pkg('kagglehub')
ensure_pkg('numpy')
ensure_pkg('pandas')
ensure_pkg('PIL', 'Pillow')
ensure_pkg('matplotlib')
ensure_pkg('cv2', 'opencv-python')
ensure_pkg('yaml', 'pyyaml')
ensure_pkg('ultralytics')
print('Dependencies ready.')

## Imports and Configuration

In [ ]:
import json, os, random, shutil
from pathlib import Path
import cv2, numpy as np, yaml
from PIL import Image
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PROJECT_DIR = Path.cwd()
WORK_DIR = PROJECT_DIR / 'working' / 'road_seg'
PREP_DIR = WORK_DIR / 'prepared_yolo_seg'
ARTIFACTS_DIR = PROJECT_DIR / 'artifacts'
for d in [WORK_DIR, PREP_DIR, ARTIFACTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

IMG_EXTS = {'.png', '.jpg', '.jpeg'}
MAX_TRAIN, MAX_VAL = 200, 50
print(f'Project dir: {PROJECT_DIR}')

## Real Kaggle Dataset Download

In [ ]:
import kagglehub

if not os.getenv('KAGGLE_USERNAME') or not os.getenv('KAGGLE_KEY'):
    raise EnvironmentError('Missing Kaggle credentials. Set KAGGLE_USERNAME and KAGGLE_KEY.')

dataset_root = Path(kagglehub.dataset_download('carlolepelaars/camvid'))
print(f'CamVid dataset root: {dataset_root}')

## Discover and Validate Road Scene Image-Mask Pairs

CamVid contains RGB images and corresponding semantic segmentation masks. We map class indices to road-related labels.

In [ ]:
def gather_files(root: Path, ext_set) -> list[Path]:
    return [p for p in root.rglob('*') if p.is_file() and p.suffix.lower() in ext_set]

all_imgs = gather_files(dataset_root, IMG_EXTS)
if len(all_imgs) < 50:
    raise RuntimeError(f'Too few images: {len(all_imgs)}')

def find_mask_for_image(img_path: Path):
    candidates = []
    stem = img_path.stem
    for mask_ext in ['_L.png', '_l.png', '_seg.png', '_mask.png']:
        candidate = img_path.parent / f'{stem}{mask_ext}'
        if candidate.exists():
            candidates.append(candidate)
    for suffix in ['label', 'mask', 'seg']:
        for p in img_path.parent.glob(f'*{suffix}*.png'):
            if stem in p.stem and p not in candidates:
                candidates.append(p)
    return candidates[0] if candidates else None

pairs = []
for img in all_imgs:
    mask = find_mask_for_image(img)
    if mask and mask.exists():
        pairs.append((img, mask))

if len(pairs) < 10:
    raise RuntimeError(f'Too few image-mask pairs: {len(pairs)}')

valid_pairs = []
for img_p, mask_p  in pairs:
    img_arr = np.array(Image.open(img_p).convert('RGB'))
    mask_arr = np.array(Image.open(mask_p).convert('L'))
    if img_arr.shape[0] < 32 or img_arr.shape[1] < 32:
        continue
    if int(mask_arr.max()) <= int(mask_arr.min()):
        continue
    valid_pairs.append((img_p, mask_p))

if len(valid_pairs) < 10:
    raise RuntimeError(f'Not enough valid pairs: {len(valid_pairs)}')

rng = random.Random(SEED)
rng.shuffle(valid_pairs)
split_idx = max(1, int(0.80 * len(valid_pairs)))
train_pairs = valid_pairs[:split_idx][:MAX_TRAIN]
val_pairs = valid_pairs[split_idx:][:MAX_VAL]

if len(val_pairs) == 0:
    val_pairs = train_pairs[-1:]
    train_pairs = train_pairs[:-1]

if len(train_pairs) == 0 or len(val_pairs) == 0:
    raise RuntimeError('Empty split.')

print(f'Valid pairs: {len(valid_pairs)}')
print(f'Train: {len(train_pairs)} | Val: {len(val_pairs)}')

In [ ]:
img0 = np.array(Image.open(train_pairs[0][0]).convert('RGB'))
mask0 = np.array(Image.open(train_pairs[0][1]).convert('L'))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].imshow(img0)
ax[0].set_title('Road Scene')
ax[0].axis('off')
ax[1].imshow(mask0, cmap='viridis')
ax[1].set_title('Semantic Segmentation Mask')
ax[1].axis('off')
plt.tight_layout()
pair_preview = ARTIFACTS_DIR / 'road_scene_sample.png'
plt.savefig(pair_preview, dpi=140)
plt.show()

## Convert Masks to YOLO Segmentation Labels

In [ ]:
for split in ['train', 'val']:
    (PREP_DIR / 'images' / split).mkdir(parents=True, exist_ok=True)
    (PREP_DIR / 'labels' / split).mkdir(parents=True, exist_ok=True)

def mask_to_yolo_segs(mask_path: Path) -> list[str]:
    m = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if m is None:
        return []
    h, w = m.shape
    _, bin_m = cv2.threshold(m, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    contours, _ = cv2.findContours(bin_m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    rows = []
    for c in contours:
        if cv2.contourArea(c) < 30:
            continue
        c = c.squeeze(1)
        if len(c.shape) != 2 or c.shape[0] < 3:
            continue
        pts = []
        for x, y in c:
            pts.extend([f'{x/w:.6f}', f'{y/h:.6f}'])
        rows.append('0 ' + ' '.join(pts))
    return rows

def process_split(pairs: list, split: str) -> tuple[int, int]:
    img_n, lbl_n = 0, 0
    for idx, (img_p, mask_p) in enumerate(pairs):
        new_name = f'{split}_{idx:05d}{img_p.suffix.lower()}'
        dst_img = PREP_DIR / 'images' / split / new_name
        dst_lbl = PREP_DIR / 'labels' / split / f'{Path(new_name).stem}.txt'
        shutil.copy2(img_p, dst_img)
        segs = mask_to_yolo_segs(mask_p)
        with open(dst_lbl, 'w', encoding='utf-8') as f:
            if segs:
                f.write('\n'.join(segs))
        img_n += 1
        if segs:
            lbl_n += 1
    return img_n, lbl_n

tr_n, tr_lbl = process_split(train_pairs, 'train')
va_n, va_lbl = process_split(val_pairs, 'val')

if tr_n == 0 or va_n == 0:
    raise RuntimeError('Empty split.')
if tr_lbl == 0 or va_lbl == 0:
    raise RuntimeError('No labeled images.')

print(f'Train: {tr_n} imgs, {tr_lbl} labeled')
print(f'Val: {va_n} imgs, {va_lbl} labeled')

In [ ]:
data_yaml = PREP_DIR / 'data.yaml'
with open(data_yaml, 'w', encoding='utf-8') as f:
    yaml.safe_dump({
        'path': str(PREP_DIR),
        'train': 'images/train',
        'val': 'images/val',
        'names': {0: 'road_scene'}
    }, f, sort_keys=False)

print(f'Data yaml: {data_yaml}')

## Train YOLO26m-seg

In [ ]:
from ultralytics import YOLO

weights = ['yolo26m-seg.pt', 'yolo11m-seg.pt', 'yolov8m-seg.pt']
selected_w = None
model = None
for w in weights:
    try:
        model = YOLO(w)
        selected_w = w
        break
    except Exception:
        pass

if selected_w is None:
    raise RuntimeError('No YOLO segmentation checkpoint available.')

print(f'Using: {selected_w}')

_ = model.train(
    data=str(data_yaml),
    epochs=2,
    imgsz=480,
    batch=4,
    project=str(ARTIFACTS_DIR / 'yolo_runs'),
    name='road_segmentation',
    seed=SEED,
    verbose=False
)

best_model_path = Path(model.trainer.best)
print(f'Best model: {best_model_path}')

## Real Evaluation

In [ ]:
best = YOLO(str(best_model_path))
val_res = best.val(data=str(data_yaml), split='val', imgsz=480, batch=4, verbose=False)

map50 = float(val_res.seg.map50)
map5095 = float(val_res.seg.map)
prec = float(val_res.seg.mp)
rec = float(val_res.seg.mr)

print(f'Road Segmentation Evaluation Results:')
print(f'  mAP50: {map50:.4f}')
print(f'  mAP50-95: {map5095:.4f}')
print(f'  Precision: {prec:.4f}')
print(f'  Recall: {rec:.4f}')

## Qualitative Predictions on Validation Set

In [ ]:
val_imgs = sorted((PREP_DIR / 'images' / 'val').iterdir())
sample_indices = list(range(min(6, len(val_imgs))))
preds = best.predict([str(val_imgs[i]) for i in sample_indices], imgsz=480, conf=0.25, verbose=False)

fig, ax = plt.subplots(2, 3, figsize=(15, 9))
for i in range(6):
    a = ax.flatten()[i]
    if i >= len(preds):
        a.axis('off')
        continue
    rendered = preds[i].plot()[:, :, ::-1]
    a.imshow(rendered)
    a.set_title(val_imgs[i].name)
    a.axis('off')
plt.tight_layout()
qual_path = ARTIFACTS_DIR / 'road_segmentation_qualitative.png'
plt.savefig(qual_path, dpi=140)
plt.show()

## Save Real Outputs

In [ ]:
metrics = {
    'dataset': 'carlolepelaars/camvid',
    'dataset_url': 'https://www.kaggle.com/datasets/carlolepelaars/camvid',
    'task': 'road_scene_segmentation',
    'train_pairs': len(train_pairs),
    'val_pairs': len(val_pairs),
    'selected_weights': selected_w,
    'best_model_path': str(best_model_path),
    'seg_map50': map50,
    'seg_map50_95': map5095,
    'seg_precision': prec,
    'seg_recall': rec
}

metrics_path = ARTIFACTS_DIR / 'metrics.json'
with open(metrics_path, 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)

manifest = {
    'pair_preview_png': str(pair_preview),
    'qualitative_predictions_png': str(qual_path),
    'metrics_json': str(metrics_path),
    'data_yaml': str(data_yaml),
    'best_model': str(best_model_path)
}
manifest_path = ARTIFACTS_DIR / 'artifact_manifest.json'
with open(manifest_path, 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2)

print('Saved:')
print(f'  Metrics: {metrics_path}')
print(f'  Manifest: {manifest_path}')
print(f'  Preview: {pair_preview}')
print(f'  Qualitative: {qual_path}')

## Limitations

- Contour-based label conversion may simplify complex scene structure.
- Single class aggregation treats all road scene elements as one; real systems need multi-class  segmentation (road, sidewalk, building, sky, tree, car, pedestrian).
- Training is intentionally short; extend epochs and data for production autonomous driving systems.
- Real deployment requires latency optimization and model quantization.